# SDG leakage audit – Colab run (profile `colab`)
**Runtime → Change runtime type → T4 GPU** before starting (GPU is used by LSTM/GRU only; trees/ridge/MLP are CPU).

**Before you start:** upload `project.zip` (from `colab_package/`) to Google Drive at `MyDrive/sdg_audit/project.zip`.

Steps: 0 mount+unpack → 1 install → 2 sanity checks → 3 smoke test → 4 grid stage A (classical) → 5 grid stage B (neural, GPU) → 6 analysis → 7 export.
Every grid cell is checkpointed to Drive: if Colab disconnects, re-run steps 0, 1 and the stage cell – finished cells are skipped.

## 0. Mount Drive and unpack

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work /content/drive/MyDrive/sdg_audit/grid_ckpt
!cd /content/work && unzip -qo /content/drive/MyDrive/sdg_audit/project.zip
%cd /content/work
# keep checkpoints in Drive so a disconnect loses nothing
!mkdir -p results/metrics && rm -rf results/metrics/colab_grid results/metrics/fast_grid && ln -sfn /content/drive/MyDrive/sdg_audit/grid_ckpt results/metrics/colab_grid
!ls

## 1. Install

In [ ]:
!pip -q install lightgbm scikit-posthocs ucimlrepo joblib tqdm pyarrow pyyaml statsmodels tabulate
# package is used via src/ on sys.path (scripts add it automatically); no pip install -e needed
import os, torch
print('CPUs:', os.cpu_count(), '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Sanity checks (unit tests + data present)

In [ ]:
!PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -8
!ls data/processed

## 3. Smoke test (≈ minutes): 24 cells on the fast profile, separate checkpoint dir

In [ ]:
!python scripts/run_full_grid.py --profile fast --n-jobs 2 --max-cells 24
import pandas as pd; d = pd.read_csv('results/metrics/fast_grid_metrics.csv'); print(d.shape); print(d['error'].notna().sum() if 'error' in d else 0, 'errors'); d.head()

## 4. Grid stage A – classical models (baselines, ridge, LightGBM, random forest)
Resumable. ≈ 3,240 cells total across stages A+B. Use `--seeds 0 1` to halve the cost if needed.

In [ ]:
!python scripts/run_full_grid.py --profile colab --n-jobs 2 --models seasonal_naive persistence ridge lightgbm random_forest mlp

## 5. Grid stage B – neural sequence models (LSTM/GRU on the T4)

In [ ]:
!python scripts/run_full_grid.py --profile colab --n-jobs 2 --models lstm gru

## 5b. Re-run P4 only (needed ONCE after the P4 row-alignment fix)
Deletes the old P4 checkpoints in Drive and recomputes just those 360 cells. Skip if you started from a fresh Drive folder.

In [ ]:
!rm -f results/metrics/colab_grid/*__P4__*.json
!python scripts/run_full_grid.py --profile colab --n-jobs 2 --protocols P4

## 6. Analysis → tables, figures, SUMMARY.md

In [ ]:
!python scripts/analyze_results.py --profile colab

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/final/figures/*.png')): display(Image(f))

## 7. Export everything to Drive and download

In [ ]:
!cd results && zip -qr /content/drive/MyDrive/sdg_audit/results_final.zip final metrics/colab_grid_metrics.csv
from google.colab import files
files.download('/content/drive/MyDrive/sdg_audit/results_final.zip')